# MTPFN on our A→B prior (with the transform class): standalone report and diagnosis

## Description

*Written 2026-11-03, before the diagnostics ran. Issue #20, branch `exp/20-mtpfn`. Arm A of notebook 12, as a standalone report. Arm R (the paper's own ICM prior) stays in notebook 12 and runs later.*

**Idea.** MTPFN (Li, Daulton, Müller, Wilson, Bakshy; SPIGM @ NeurIPS 2025) is the published PFN for in-context transfer between tasks.
- **Data.** Every task's points form a set. Each task also gets a [TASK] summary token.
- **Within a task** (intra-task attention): points and the [TASK] token attend to each other. The target's query points read only the target's own tokens.
- **Between tasks** (inter-task attention): only the [TASK] tokens attend to each other, so all transfer passes through one vector per task and layer.
- **Training.** The paper trains it on a robust ICM prior, where tasks share a lengthscale and correlate; with probability p a source task is unrelated.

**What we ask here.** Train the same architecture on *our* prior, the pilot-3 AIR prior:
- B: a dense source (128 points, noise σ_B);
- A: a scarce target (n_A ∈ 1..10);
- f_A(x) = g(T_θ(x)), with g a random MLP on the global ECDF scale;
- T_θ a monotone, box-preserving flow from the transform class, J ∈ {2, 4, 8, 16} coefficients;
- H0 (h = id) and Z0 (always related).

Then ask:
1. **Transfer.** How much better than A alone is it?
2. **Gap.** How far is it from the exact dense-B Bayes predictive (G-ref, J = 2), and from AIR (pilot 3, when its evaluation is in)?
3. **Diagnosis.** Why does it lose what it loses? Concretely: does it register B to A, that is, read B at T_θ(x*)?

**Setup.**
- **Model.** Width 128, N = 4 [intra, inter] pairs plus a final intra block, 4 heads, d_ff 512. That is 1.84M parameters, the AIR decoder + proposal budget, far below the paper's 23 × 512.
- **Training.** Bar output with AIR's bins (400 bins on [−0.25, 1.25]), so NLLs are on the same scale as AIR's. AdamW 1e-4, cosine, 100k steps × 64 tasks, seeds 0, 1, 2 (sweep `runs/mtpfn/pilot_20261103-132611`).
- **Evaluation.** Pilot 3's evaluation tasks: `eval_tasks(J, 256, seed 777, ecdf)`, J ∈ {2, 16}, n_A ∈ {1, 2, 3, 5, 10}, nested A designs.
- **References** on the same tasks:
  - pilot 3's clean A-only PFN, same seed (`aonly_pfn`);
  - G-ref at J = 2 (`gref`): exact Bayes given g, so a floor rather than a target.

**Diagnostics** (`experimental/mtpfn/diagnose.py`, mode `air_diag`, one run per seed):
- **D1 Transfer.** NLL(`aonly_pfn`) − NLL(MTPFN) per (J, n_A).
- **D2 Gap.** NLL(MTPFN) − NLL(`gref`), J = 2.
- **D3 Input controls** on the same tasks:
  - `b_registered`: B pulled into A's frame with the true θ. The gain over `b` is what MTPFN loses by having to handle the warp itself.
  - `b_other`: an unrelated B, i.e. another task's B.
  - `b_none`: B with only its [TASK] token, which is ≈ A-only and off-distribution.
- **D4 Where does it read B?**
  - Compute the sensitivity s_k = |∂E[y*|x*]/∂y_{B,k}| for every B point.
  - Summarise it as `read`, the sensitivity-weighted mean x_B, and `conc`, the share of sensitivity within ±0.05 of T_θ(x*).
  - Compare |read − T_θ(x*)| with |x* − T_θ(x*)|, i.e. with reading at x* itself.
- **D5 Calibration.** PIT of y* (128 tasks × 32 queries per cell).
- **D6 Warp size.** The D1 gain against the task's mean |T_θ(x) − x|.
- **Examples.** PPDs (mean and 5–95 % quantiles) on an x grid next to f_A, B, A and the G-ref predictive.

**Reading rules (pre-registered).** n_A ≥ 3, averaged over tasks; means over 3 seeds ± SE where that applies:
- **"MTPFN registers"**: median `conc` ≥ 0.5 and median |read − T| < median |x* − T|.
- **"It reads globally (summary transfer)"**: median `conc` < 0.2 and `read` ≈ 0.5 (|read − 0.5| < 0.1 for most x*).
- **"The warp is its bottleneck"**: NLL(`b`) − NLL(`b_registered`) ≥ 0.1 at n_A ≥ 2.
- **"Uses B"**: NLL(`b_other`) − NLL(`b`) ≥ 0.1.
- **Calibration**: reported as the PIT histogram plus the KS distance to uniform, no threshold.

**Expectation, written before the run.** MTPFN can pass B to A only through [TASK] summaries. So we expect it to use B (global level, scale, shape statistics), with a real gain over A-only at small n_A. But we expect it not to register B pointwise: low `conc`, a large `b_registered` gain, and a large gap to G-ref and to AIR, growing with J.

**Limits.** d = 1, Z0, H0 (pilot 3's scope). The model is smaller than the paper's. G-ref knows g exactly. Sensitivity is local (gradient), so it shows where the predictive *mean* reads, not the whole PPD.

## Results

*Filled in when the diagnostics (`task.mode=air_diag`, one run per seed) are done.*